# Program 10: Trade Journal Analytics Engine

**Quantitative Trade Performance Analysis**

This notebook reads your trade history from a CSV (or auto-generates realistic sample data) and calculates every performance metric a professional trader needs:

- **Equity curve** with underwater drawdown overlay
- **Rolling Sharpe ratio** (risk-adjusted return quality over time)
- **Win rate, avg win/loss, profit factor** broken down by strategy
- **Behavioral heatmap**: P&L by day-of-week and entry hour
- **Holding period distribution** and strategy comparison

The goal: find your **actual edge** and your **blind spots**.

---

**Poker analogy**: Your trade journal is your hand history database. Without tracking results rigorously, you are guessing about your win rate. This program is your PokerTracker for options.

In [ ]:
# ============================================================
# Setup: Install dependencies and import libraries
# ============================================================
!pip install numpy pandas matplotlib seaborn -q

import warnings
warnings.filterwarnings('ignore')

import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from datetime import datetime, date, timedelta
import random

print('All imports loaded successfully.')

## 1. Configuration

Set the path to your trade journal CSV and key parameters.

**CSV format**: `date, close_date, hour, ticker, strategy, direction, entry_price, exit_price, contracts, hold_days, pnl`

If the CSV does not exist, sample data is auto-generated with realistic P&L distributions for 7 options strategies.

In [ ]:
# ============================================================
# Configuration
# ============================================================

JOURNAL_CSV_PATH = "trade_journal.csv"    # path to your CSV; auto-generated if missing
ROLLING_SHARPE_WINDOW = 30                # trading days for rolling Sharpe
RISK_FREE_DAILY = 0.0525 / 252            # annual risk-free rate / 252

# Strategies expected (used for color-coding and filtering)
STRATEGIES = [
    "iron_condor",
    "covered_call",
    "long_call",
    "long_put",
    "cash_secured_put",
    "straddle",
    "vertical_spread",
]

STRATEGY_COLORS = {
    "iron_condor":      "#3498db",
    "covered_call":     "#2ecc71",
    "long_call":        "#f39c12",
    "long_put":         "#e74c3c",
    "cash_secured_put": "#9b59b6",
    "straddle":         "#1abc9c",
    "vertical_spread":  "#e67e22",
}

print(f'Sharpe window: {ROLLING_SHARPE_WINDOW} days')
print(f'Risk-free daily rate: {RISK_FREE_DAILY:.6f}')
print(f'Strategies tracked: {len(STRATEGIES)}')

## 2. Sample Data Generator

Each strategy has distinct statistical properties that mirror real-world options trading:

| Strategy | Win Rate | Avg Win | Avg Loss | Character |
|---|---|---|---|---|
| Iron Condor | 72% | $280 | -$420 | High win rate, big tail losses |
| Covered Call | 68% | $180 | -$250 | Steady income, capped upside |
| Long Call | 38% | $850 | -$290 | Low win rate, outsized wins |
| Straddle | 45% | $620 | -$380 | Needs big moves to profit |

**Key insight**: High win rate does NOT mean profitable. What matters is **expected value**:

$$E[\text{trade}] = P(\text{win}) \times \overline{W} + P(\text{loss}) \times \overline{L}$$

A 38% win rate strategy (long call) can be more profitable than a 72% win rate strategy (iron condor) if the avg win is large enough relative to the avg loss.

In [ ]:
# ============================================================
# Sample Data Generator
# ============================================================

def generate_sample_journal(n_trades=180, output_path=JOURNAL_CSV_PATH):
    """
    Generate realistic sample trade journal data and save to CSV.
    Simulates 6 months of options trading with realistic P&L distributions.
    """
    print(f"  Generating {n_trades} sample trades -> {output_path}")
    random.seed(42)
    np.random.seed(42)

    tickers    = ["SPY", "QQQ", "AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "META"]
    directions = ["long", "short"]

    # Each strategy has different win rate and avg P&L characteristics
    strategy_params = {
        "iron_condor":      {"win_rate": 0.72, "avg_win": 280, "avg_loss": -420, "freq": 0.25},
        "covered_call":     {"win_rate": 0.68, "avg_win": 180, "avg_loss": -250, "freq": 0.18},
        "cash_secured_put": {"win_rate": 0.70, "avg_win": 200, "avg_loss": -380, "freq": 0.18},
        "long_call":        {"win_rate": 0.38, "avg_win": 850, "avg_loss": -290, "freq": 0.12},
        "long_put":         {"win_rate": 0.35, "avg_win": 700, "avg_loss": -260, "freq": 0.08},
        "straddle":         {"win_rate": 0.45, "avg_win": 620, "avg_loss": -380, "freq": 0.10},
        "vertical_spread":  {"win_rate": 0.58, "avg_win": 320, "avg_loss": -280, "freq": 0.09},
    }

    strategy_list = list(strategy_params.keys())
    cum_freq = np.cumsum([strategy_params[s]['freq'] for s in strategy_list])

    start_date = date.today() - timedelta(days=200)
    rows = []

    for _ in range(n_trades):
        r = random.random()
        strat = strategy_list[np.searchsorted(cum_freq, r)]
        params = strategy_params[strat]

        offset = random.randint(0, 195)
        trade_date = start_date + timedelta(days=offset)
        while trade_date.weekday() >= 5:
            trade_date += timedelta(days=1)

        entry_hour = random.choices(
            [9, 10, 11, 12, 13, 14, 15],
            weights=[0.25, 0.18, 0.12, 0.08, 0.12, 0.15, 0.10])[0]

        ticker = random.choice(tickers)

        win = random.random() < params['win_rate']
        if win:
            pnl = abs(np.random.normal(params['avg_win'], params['avg_win'] * 0.4))
        else:
            pnl = -abs(np.random.normal(abs(params['avg_loss']), abs(params['avg_loss']) * 0.35))
        pnl = round(pnl, 2)

        base_price = {"SPY": 540, "QQQ": 470, "AAPL": 225, "NVDA": 880,
                      "MSFT": 410, "AMZN": 195, "TSLA": 270, "META": 555}.get(ticker, 100)
        entry_px = round(base_price * (1 + np.random.normal(0, 0.02)), 2)
        exit_px  = round(entry_px + (pnl / (abs(pnl) * 0.1 + 1)) * np.random.uniform(0.5, 2.0), 2)
        contracts = random.choice([1, 1, 1, 2, 2, 3])
        direction = "short" if strat in ["iron_condor", "covered_call", "cash_secured_put"] else "long"

        if strat in ["iron_condor", "covered_call", "cash_secured_put"]:
            hold_days = random.randint(14, 45)
        elif strat in ["long_call", "long_put"]:
            hold_days = random.randint(1, 21)
        else:
            hold_days = random.randint(5, 30)

        close_date = trade_date + timedelta(days=hold_days)

        rows.append({
            "date":        trade_date.strftime('%Y-%m-%d'),
            "close_date":  close_date.strftime('%Y-%m-%d'),
            "hour":        entry_hour,
            "ticker":      ticker,
            "strategy":    strat,
            "direction":   direction,
            "entry_price": entry_px,
            "exit_price":  exit_px,
            "contracts":   contracts,
            "hold_days":   hold_days,
            "pnl":         pnl,
        })

    df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
    df.to_csv(output_path, index=False)
    print(f"  Sample journal saved ({len(df)} trades).")
    return df

## 3. Load Journal & Core Metrics

### Equity Curve & Drawdown

The equity curve is the cumulative sum of P&L. The **drawdown** at any point is how far you have fallen from the peak:

$$\text{Drawdown}_t = \text{Equity}_t - \max_{s \leq t} \text{Equity}_s$$

Max drawdown tells you the worst peak-to-trough decline. This is the number that determines whether you can psychologically survive your strategy.

### Rolling Sharpe Ratio

The Sharpe ratio measures risk-adjusted return:

$$S = \frac{E[R - R_f]}{\sigma_R} \times \sqrt{252}$$

where $R$ is daily P&L return, $R_f$ is the risk-free rate, and $\sigma_R$ is standard deviation. We annualize by multiplying by $\sqrt{252}$ (trading days per year).

- **Sharpe > 1.0**: Good risk-adjusted performance
- **Sharpe > 2.0**: Excellent (most hedge funds target this)
- **Sharpe < 0**: You are losing money on a risk-adjusted basis

In [ ]:
# ============================================================
# Load Journal & Core Metrics
# ============================================================

def load_journal(path=JOURNAL_CSV_PATH):
    """Load CSV or generate sample if missing."""
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['date'])
        print(f"  Loaded {len(df)} trades from {path}")
    else:
        print(f"  {path} not found -- generating sample data...")
        df = generate_sample_journal(output_path=path)
        df['date'] = pd.to_datetime(df['date'])
    required = ['date', 'ticker', 'strategy', 'direction', 'pnl']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"CSV missing required column: {col}")
    if 'hold_days' not in df.columns:
        df['hold_days'] = 1
    if 'hour' not in df.columns:
        df['hour'] = 10
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    return df


def compute_equity_curve(df):
    """Cumulative P&L and drawdown series."""
    equity = df.set_index('date')['pnl'].cumsum()
    rolling_max = equity.cummax()
    drawdown = equity - rolling_max
    return equity, drawdown


def rolling_sharpe(df, window=ROLLING_SHARPE_WINDOW):
    """Rolling Sharpe ratio (excess return / vol)."""
    daily = df.groupby('date')['pnl'].sum()
    daily_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='B')
    daily = daily.reindex(daily_idx, fill_value=0.0)

    def sharpe_window(returns):
        excess = returns - RISK_FREE_DAILY
        if excess.std() == 0:
            return np.nan
        return (excess.mean() / excess.std()) * np.sqrt(252)

    return daily.rolling(window).apply(sharpe_window, raw=True)


# Load the data
print("Loading trade journal...")
df = load_journal(JOURNAL_CSV_PATH)

print("\nComputing equity curve and drawdown...")
equity, drawdown = compute_equity_curve(df)
print(f"  Final equity: ${equity.iloc[-1]:,.0f}")
print(f"  Max drawdown: ${drawdown.min():,.0f}")

print("\nComputing rolling Sharpe ratio...")
sharpe_series = rolling_sharpe(df)
last_sharpe = sharpe_series.dropna().iloc[-1] if len(sharpe_series.dropna()) > 0 else np.nan
print(f"  Current {ROLLING_SHARPE_WINDOW}d Sharpe: {last_sharpe:.2f}")

## 4. Strategy Breakdown & Profit Factor

### Profit Factor

Profit Factor is the ratio of gross profits to gross losses:

$$\text{Profit Factor} = \frac{\sum \text{Winning Trades}}{|\sum \text{Losing Trades}|}$$

- **PF > 1.0**: Making money (gross wins > gross losses)
- **PF > 1.5**: Strong edge
- **PF > 2.0**: Exceptional (rare to sustain)
- **PF < 1.0**: Losing money -- review immediately

### Why Per-Strategy Breakdown Matters

Your overall stats may look fine, but one strategy could be dragging you down. Breaking down by strategy reveals which setups actually have edge and which are bleeding capital.

In [ ]:
# ============================================================
# Strategy Breakdown
# ============================================================

def strategy_breakdown(df):
    """Per-strategy performance metrics."""
    rows = []
    for strat in df['strategy'].unique():
        sub = df[df['strategy'] == strat]
        wins  = sub[sub['pnl'] > 0]
        losses = sub[sub['pnl'] <= 0]
        win_rate = len(wins) / len(sub) if len(sub) > 0 else 0
        avg_win  = wins['pnl'].mean()  if len(wins)  > 0 else 0
        avg_loss = losses['pnl'].mean() if len(losses) > 0 else 0
        profit_factor = (wins['pnl'].sum() / abs(losses['pnl'].sum())
                         if abs(losses['pnl'].sum()) > 0 else np.inf)
        total_pnl = sub['pnl'].sum()
        rows.append({
            "strategy":      strat,
            "n_trades":      len(sub),
            "win_rate":      win_rate,
            "avg_win":       avg_win,
            "avg_loss":      avg_loss,
            "profit_factor": profit_factor,
            "total_pnl":     total_pnl,
        })
    return pd.DataFrame(rows).sort_values('total_pnl', ascending=False)


strat_df = strategy_breakdown(df)

print("Strategy Breakdown:")
print(f"{'Strategy':<20} {'Trades':>6} {'WinRate':>8} {'PF':>6} {'Total P&L':>10}")
print("-" * 55)
for _, row in strat_df.iterrows():
    print(f"{row['strategy']:<20} {row['n_trades']:>6} "
          f"{row['win_rate']*100:>7.1f}% {min(row['profit_factor'],99):>6.2f} "
          f"${row['total_pnl']:>9,.0f}")

## 5. Behavioral Analysis

### Trading Psychology Leaks

Most traders lose money not because of bad strategy, but because of **behavioral biases**:

- **Monday tilt**: Overtrading at open after a weekend of analysis paralysis
- **Friday FOMO**: Entering positions late in the week before theta weekend decay
- **Opening hour aggression**: Market open (9:30-10:00) has the widest spreads and highest noise
- **Afternoon fatigue**: Decision quality degrades after hours of screen time

By mapping P&L to day-of-week and entry hour, we can detect these patterns and **eliminate the leak**.

In [ ]:
# ============================================================
# Behavioral Analysis
# ============================================================

def behavioral_analysis(df):
    """P&L by day-of-week and hour."""
    df = df.copy()
    df['day_of_week'] = df['date'].dt.dayofweek   # 0=Mon
    df['hour'] = df['hour'].fillna(10).astype(int)

    dow_pnl = df.groupby('day_of_week')['pnl'].agg(['sum', 'mean', 'count'])
    dow_pnl.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'][:len(dow_pnl)]

    hour_pnl = df.groupby('hour')['pnl'].agg(['sum', 'mean', 'count'])

    return dow_pnl, hour_pnl


def overall_stats(df):
    """Compute headline statistics."""
    wins   = df[df['pnl'] > 0]
    losses = df[df['pnl'] <= 0]
    total  = len(df)
    equity, dd = compute_equity_curve(df)
    sharpe_ser = rolling_sharpe(df)
    final_sharpe = sharpe_ser.iloc[-1] if len(sharpe_ser) > 0 else np.nan
    max_dd = dd.min()
    return {
        "total_trades":    total,
        "win_rate":        len(wins) / total if total > 0 else 0,
        "avg_win":         wins['pnl'].mean() if len(wins) > 0 else 0,
        "avg_loss":        losses['pnl'].mean() if len(losses) > 0 else 0,
        "profit_factor":   (wins['pnl'].sum() / abs(losses['pnl'].sum())
                            if abs(losses['pnl'].sum()) > 0 else np.inf),
        "total_pnl":       df['pnl'].sum(),
        "max_drawdown":    max_dd,
        "sharpe":          final_sharpe,
        "best_trade":      df['pnl'].max(),
        "worst_trade":     df['pnl'].min(),
        "avg_hold_days":   df['hold_days'].mean() if 'hold_days' in df.columns else np.nan,
    }


dow_pnl, hour_pnl = behavioral_analysis(df)
stats = overall_stats(df)

print("Behavioral Analysis:")
print(f"  Best day:  {dow_pnl['sum'].idxmax()}")
print(f"  Worst day: {dow_pnl['sum'].idxmin()}")
print(f"\nOverall Stats:")
print(f"  Win Rate:       {stats['win_rate']*100:.1f}%")
print(f"  Profit Factor:  {min(stats['profit_factor'],99):.2f}")
print(f"  Total P&L:      ${stats['total_pnl']:,.0f}")
print(f"  Max Drawdown:   ${stats['max_drawdown']:,.0f}")
print(f"  Sharpe Ratio:   {stats['sharpe']:.2f}" if not np.isnan(stats['sharpe']) else "  Sharpe: N/A")

## 6. Full Dashboard Visualization

The 8-panel dashboard brings everything together:

1. **Equity Curve + Drawdown** -- your P&L trajectory and pain points
2. **Rolling Sharpe** -- is your edge consistent or deteriorating?
3. **Strategy P&L** -- which strategies actually make money?
4. **Win Rate + Profit Factor** -- quality vs quantity by strategy
5. **Day-of-Week P&L** -- behavioral timing leaks
6. **Entry Hour P&L** -- intraday timing leaks
7. **Holding Period Distribution** -- are you closing too early/late?
8. **Performance Summary** -- headline numbers at a glance

In [ ]:
# ============================================================
# Full Dashboard Visualization
# ============================================================

def plot_dashboard(df, equity, drawdown, sharpe_series, strat_df,
                   dow_pnl, hour_pnl, stats):
    plt.style.use('dark_background')
    fig = plt.figure(figsize=(20, 18), facecolor='#0d1117')
    fig.suptitle("Trade Journal Analytics Engine",
                 fontsize=16, fontweight='bold', color='white', y=0.99)

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38,
                           top=0.95, bottom=0.05, left=0.06, right=0.97)

    PLOT_BG = '#151b27'
    color_pos = '#2ecc71'
    color_neg = '#e74c3c'

    # -- Panel 1: Equity Curve + Drawdown --
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.set_facecolor(PLOT_BG)
    ax1.plot(equity.index, equity.values, color=color_pos, linewidth=2,
             label='Equity Curve', zorder=3)
    ax1.fill_between(equity.index, equity.values, 0,
                     where=equity.values >= 0, color=color_pos, alpha=0.15)
    ax1.fill_between(equity.index, equity.values, 0,
                     where=equity.values < 0, color=color_neg, alpha=0.2)
    ax1.axhline(0, color='#555', linewidth=0.8)

    ax2_twin = ax1.twinx()
    ax2_twin.fill_between(drawdown.index, drawdown.values, 0,
                          color='#e74c3c', alpha=0.35, label='Drawdown')
    ax2_twin.set_ylabel("Drawdown ($)", color='#e74c3c', fontsize=8)
    ax2_twin.tick_params(colors='#e74c3c', labelsize=7)

    ax1.set_title("Equity Curve & Drawdown", color='white', fontsize=11,
                  fontweight='bold', pad=8)
    ax1.set_ylabel("Cumulative P&L ($)", color='#aaa', fontsize=9)
    ax1.tick_params(colors='#aaa', labelsize=8)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax1.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')
    final_pnl = equity.iloc[-1] if len(equity) > 0 else 0
    ax1.text(0.98, 0.92, f"Total P&L: ${final_pnl:,.0f}",
             transform=ax1.transAxes, ha='right', color='white',
             fontsize=11, fontweight='bold')
    for spine in ax1.spines.values():
        spine.set_edgecolor('#333')

    # -- Panel 2: Rolling Sharpe --
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.set_facecolor(PLOT_BG)
    sharpe_clean = sharpe_series.dropna()
    if len(sharpe_clean) > 0:
        ax2.plot(sharpe_clean.index, sharpe_clean.values, color='#3498db', linewidth=1.5)
        ax2.axhline(0, color='#555', linewidth=0.8)
        ax2.axhline(1, color='#2ecc71', linewidth=1, linestyle='--', alpha=0.7, label='Sharpe=1')
        ax2.fill_between(sharpe_clean.index, sharpe_clean.values, 0,
                         where=sharpe_clean.values >= 0, color='#3498db', alpha=0.15)
        ax2.fill_between(sharpe_clean.index, sharpe_clean.values, 0,
                         where=sharpe_clean.values < 0, color='#e74c3c', alpha=0.2)
        last_s = sharpe_clean.iloc[-1]
        ax2.text(0.98, 0.92, f"Current: {last_s:.2f}",
                 transform=ax2.transAxes, ha='right', color='white',
                 fontsize=10, fontweight='bold')
    ax2.set_title(f"Rolling {ROLLING_SHARPE_WINDOW}d Sharpe Ratio", color='white',
                  fontsize=10, fontweight='bold', pad=8)
    ax2.set_ylabel("Sharpe", color='#aaa', fontsize=8)
    ax2.tick_params(colors='#aaa', labelsize=7)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    for spine in ax2.spines.values():
        spine.set_edgecolor('#333')

    # -- Panel 3: Strategy P&L Comparison --
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_facecolor(PLOT_BG)
    strats  = strat_df['strategy'].values
    totals  = strat_df['total_pnl'].values
    s_colors = [STRATEGY_COLORS.get(s, '#aaa') for s in strats]
    bars = ax3.barh(strats, totals, color=s_colors, edgecolor='#333', height=0.6)
    ax3.axvline(0, color='white', linewidth=1, alpha=0.5)
    for bar, val in zip(bars, totals):
        x = val + (max(abs(totals)) * 0.02 * np.sign(val))
        ax3.text(x, bar.get_y() + bar.get_height() / 2,
                 f"${val:,.0f}", va='center', color='white', fontsize=8)
    ax3.set_title("Total P&L by Strategy", color='white', fontsize=10,
                  fontweight='bold', pad=8)
    ax3.set_xlabel("P&L ($)", color='#aaa', fontsize=8)
    ax3.tick_params(colors='#ccc', labelsize=8)
    for spine in ax3.spines.values():
        spine.set_edgecolor('#333')

    # -- Panel 4: Win Rate & Profit Factor --
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_facecolor(PLOT_BG)
    x = np.arange(len(strat_df))
    width = 0.35
    wr_bars = ax4.bar(x - width/2, strat_df['win_rate'] * 100, width,
                      color='#3498db', alpha=0.85, label='Win Rate %', edgecolor='#333')
    ax4.axhline(50, color='#555', linewidth=0.8, linestyle='--')
    ax4_twin = ax4.twinx()
    pf_vals = np.clip(strat_df['profit_factor'].replace([np.inf, -np.inf], 5.0), 0, 5)
    pf_bars = ax4_twin.bar(x + width/2, pf_vals, width,
                            color='#2ecc71', alpha=0.85, label='Profit Factor',
                            edgecolor='#333')
    ax4_twin.axhline(1.0, color='#e74c3c', linewidth=1, linestyle='--', alpha=0.7)
    ax4_twin.set_ylabel("Profit Factor", color='#2ecc71', fontsize=8)
    ax4_twin.tick_params(colors='#2ecc71', labelsize=7)
    ax4.set_xticks(x)
    ax4.set_xticklabels(strat_df['strategy'], rotation=35, ha='right',
                        color='#ccc', fontsize=7)
    ax4.set_title("Win Rate & Profit Factor by Strategy", color='white',
                  fontsize=10, fontweight='bold', pad=8)
    ax4.set_ylabel("Win Rate (%)", color='#3498db', fontsize=8)
    ax4.tick_params(colors='#aaa', labelsize=7)
    for spine in ax4.spines.values():
        spine.set_edgecolor('#333')

    # -- Panel 5: Day of Week --
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.set_facecolor(PLOT_BG)
    dow_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in dow_pnl['sum'].values]
    ax5.bar(dow_pnl.index, dow_pnl['sum'].values, color=dow_colors, edgecolor='#333', width=0.6)
    ax5.axhline(0, color='white', linewidth=0.8, alpha=0.5)
    ax5.set_title("P&L by Day of Week", color='white', fontsize=10,
                  fontweight='bold', pad=8)
    ax5.set_ylabel("Total P&L ($)", color='#aaa', fontsize=8)
    ax5.tick_params(colors='#ccc', labelsize=9)
    for spine in ax5.spines.values():
        spine.set_edgecolor('#333')
    best_day = dow_pnl['sum'].idxmax()
    ax5.text(0.5, 0.92, f"Best day: {best_day}", transform=ax5.transAxes,
             ha='center', color='#2ecc71', fontsize=9)

    # -- Panel 6: Entry Hour --
    ax6 = fig.add_subplot(gs[2, 0])
    ax6.set_facecolor(PLOT_BG)
    hour_pnl_sorted = hour_pnl.sort_index()
    h_colors = ['#2ecc71' if v >= 0 else '#e74c3c'
                for v in hour_pnl_sorted['sum'].values]
    ax6.bar([f"{h:02d}:00" for h in hour_pnl_sorted.index],
            hour_pnl_sorted['sum'].values, color=h_colors, edgecolor='#333', width=0.6)
    ax6.axhline(0, color='white', linewidth=0.8, alpha=0.5)
    ax6.set_title("P&L by Entry Hour", color='white', fontsize=10,
                  fontweight='bold', pad=8)
    ax6.set_ylabel("Total P&L ($)", color='#aaa', fontsize=8)
    ax6.tick_params(colors='#ccc', labelsize=8)
    plt.setp(ax6.xaxis.get_majorticklabels(), rotation=45, ha='right')
    for spine in ax6.spines.values():
        spine.set_edgecolor('#333')

    # -- Panel 7: Holding Period Distribution --
    ax7 = fig.add_subplot(gs[2, 1])
    ax7.set_facecolor(PLOT_BG)
    if 'hold_days' in df.columns:
        hold = df['hold_days'].dropna()
        ax7.hist(hold.values, bins=20, color='#9b59b6', edgecolor='#333', alpha=0.85)
        ax7.axvline(hold.mean(), color='white', linewidth=1.5, linestyle='--',
                    label=f'Mean={hold.mean():.1f}d')
        ax7.legend(fontsize=8, facecolor='#1a1f2e', labelcolor='white')
    ax7.set_title("Holding Period Distribution", color='white', fontsize=10,
                  fontweight='bold', pad=8)
    ax7.set_xlabel("Days Held", color='#aaa', fontsize=8)
    ax7.set_ylabel("# Trades", color='#aaa', fontsize=8)
    ax7.tick_params(colors='#aaa', labelsize=8)
    for spine in ax7.spines.values():
        spine.set_edgecolor('#333')

    # -- Panel 8: Performance Summary --
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.set_facecolor(PLOT_BG)
    ax8.axis('off')
    ax8.set_title("Performance Summary", color='white', fontsize=10,
                  fontweight='bold', pad=8)

    summary = [
        ("Total Trades",     f"{stats['total_trades']}"),
        ("Win Rate",         f"{stats['win_rate']*100:.1f}%"),
        ("Avg Win",          f"${stats['avg_win']:,.0f}"),
        ("Avg Loss",         f"${stats['avg_loss']:,.0f}"),
        ("Profit Factor",    f"{min(stats['profit_factor'], 99):.2f}"),
        ("Total P&L",        f"${stats['total_pnl']:,.0f}"),
        ("Max Drawdown",     f"${stats['max_drawdown']:,.0f}"),
        ("Sharpe (rolling)", f"{stats['sharpe']:.2f}" if not np.isnan(stats['sharpe']) else "N/A"),
        ("Best Trade",       f"${stats['best_trade']:,.0f}"),
        ("Worst Trade",      f"${stats['worst_trade']:,.0f}"),
        ("Avg Hold (days)",  f"{stats['avg_hold_days']:.1f}" if not np.isnan(stats['avg_hold_days']) else "N/A"),
    ]

    for j, (label, value) in enumerate(summary):
        y = 0.95 - j * 0.087
        if any(x in label for x in ["P&L", "Win", "Loss", "Factor", "Sharpe", "Drawdown"]):
            try:
                num = float(value.replace('$', '').replace('%', '').replace(',', ''))
                color = '#2ecc71' if num > 0 else '#e74c3c' if num < 0 else 'white'
            except Exception:
                color = 'white'
        else:
            color = '#ccc'
        ax8.text(0.05, y, label, transform=ax8.transAxes, color='#888', fontsize=9)
        ax8.text(0.95, y, value, transform=ax8.transAxes, ha='right',
                 color=color, fontsize=9, fontweight='bold')
        ax8.axhline(y=y - 0.035, xmin=0.03, xmax=0.97, color='#333',
                    linewidth=0.5, transform=ax8.transAxes)

    plt.savefig("journal_analytics.png", dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    print("Dashboard saved to journal_analytics.png")
    plt.show()


# Render the full dashboard
print("Rendering dashboard...")
plot_dashboard(df, equity, drawdown, sharpe_series, strat_df,
               dow_pnl, hour_pnl, stats)
print("Done.")

---

## Exercise 1: Modify Strategy Parameters

In the sample data generator, change the `iron_condor` win rate from 72% to 55% and re-run the notebook.

**Questions to answer**:
1. How does the profit factor change for iron condors?
2. Does it still contribute positive P&L, or does the avg loss overwhelm the lower win rate?
3. At what win rate does the iron condor strategy break even, given avg win = $280 and avg loss = -$420?

**Hint**: Breakeven win rate = $|\text{avg loss}| / (\text{avg win} + |\text{avg loss}|)$ = $420 / (280 + 420) = 60\%$

## Exercise 2: Sharpe Window Sensitivity

Change `ROLLING_SHARPE_WINDOW` from 30 to 10, then to 60. Re-run the notebook.

**Observe**:
1. Shorter windows (10d) produce noisier Sharpe estimates -- more spikes but less reliable
2. Longer windows (60d) are smoother but lag -- they miss recent regime changes
3. Which window gives you the most actionable signal for adjusting your strategy?

## Exercise 3: Load Your Own Trade Data

Export your real trades to a CSV with these columns:

```
date, close_date, hour, ticker, strategy, direction, entry_price, exit_price, contracts, hold_days, pnl
```

Upload to Colab and set `JOURNAL_CSV_PATH` to point to your file. Re-run and examine:

1. Which strategy has the highest profit factor?
2. Which day of the week are you weakest?
3. Is your Sharpe above 1.0? If not, which strategy drags it down?

---

## Key Takeaways

1. **Win rate alone is meaningless** -- what matters is Expected Value = P(win) x avg_win + P(loss) x avg_loss. A 38% win rate strategy can outperform a 72% win rate strategy.

2. **Profit Factor > 1.5 is the target** -- this means for every $1 you lose, you make $1.50. Below 1.0 and you are donating money.

3. **Sharpe ratio is the gold standard** -- it adjusts returns for risk. A high-return, high-volatility strategy may have a lower Sharpe than a steady lower-return strategy.

4. **Behavioral leaks are real** -- most traders have statistically significant P&L differences by day-of-week and entry hour. Finding and eliminating these is free alpha.

5. **Max drawdown determines survival** -- your strategy may be profitable in expectation, but if the max drawdown exceeds your psychological or financial tolerance, you will quit at the worst possible time.

6. **Track everything** -- the trade journal is the single most important tool for improvement. Without data, you are gambling. With data, you are a professional.